# Pixel-cluster BDT baseline

This notebook is the optional BDT/background notebook for the tutorial. It shows the traditional feature-based classifier:

1. load the same signal/BIB cluster samples,
2. build a fixed-size table of cluster-level variables,
3. train or load a Gradient Boosted Decision Tree,
4. evaluate its score distribution, ROC curve, confusion matrix, and feature importance.

The Transformer tutorial notebook will use the BDT only as a reference point.

In [ ]:
import sys, os, json, math, time, random
from pathlib import Path

NOTEBOOK_DIR = '/global/cfs/cdirs/m5197/sferrar2/ML4FPS/Jupyter'
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

import h5py
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from dataloader import load_pixel_data, CLUSTER_FEATURE_KEYS
from tutorial_utils import (
    plot_feature_distributions, plot_feature_importance,
    plot_confusion_matrix, plot_roc_comparison, plot_leaderboard,
    plot_score_distributions, evaluate_classifier, benchmark_bdt,
    print_benchmark,
)

## Configuration

In [ ]:
SIGNAL_H5 = '../Samples_v2/signal.h5'
BIB_H5    = '../Samples_v2/BIB.h5'

MAX_RAW_HITS = 50
TEST_FRACTION = 0.2
VAL_FRACTION  = 0.1
RANDOM_SEED   = 42
BATCH_SIZE    = 256

OUTPUT_DIR = Path('bdt_baseline_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BDT_MODEL_PATH = OUTPUT_DIR / 'bdt_gradient_boosting.joblib'

## Load the same train/validation/test split

We use `load_pixel_data(...)` mostly to get the same labels and split indices used by the Transformer notebooks. The BDT itself will use the fixed-size cluster features from the HDF5 files.

In [ ]:
data = load_pixel_data(
    SIGNAL_H5, BIB_H5,
    max_hits=MAX_RAW_HITS,
    batch=BATCH_SIZE,
    test_frac=TEST_FRACTION,
    val_frac=VAL_FRACTION,
    seed=RANDOM_SEED,
)

idx_train = np.asarray(data['idx_train'])
idx_val   = np.asarray(data['idx_val'])
idx_test  = np.asarray(data['idx_test'])
labels    = np.asarray(data['labels']).astype(int)

print(f"Total clusters: {len(labels):,}")
print(f"Signal: {labels.sum():,}, BIB: {(labels == 0).sum():,}")
print(f"Train: {len(idx_train):,}, Val: {len(idx_val):,}, Test: {len(idx_test):,}")

## Load fixed-size BDT features

In [ ]:
def as_feature_key_list(keys):
    if isinstance(keys, dict):
        keys = list(keys.keys())
    keys = [str(k) for k in keys]
    return keys

FALLBACK_BDT_KEYS = [
    'cluster_energy', 'cluster_time', 'cluster_r', 'incident_angle',
    'cluster_size_tot', 'cluster_size_x', 'cluster_size_y',
    'cluster_rms_x', 'cluster_rms_y', 'cluster_skew_x', 'cluster_skew_y',
    'cluster_aspect', 'cluster_ecc',
]

try:
    BDT_FEATURE_KEYS = as_feature_key_list(CLUSTER_FEATURE_KEYS)
except Exception:
    BDT_FEATURE_KEYS = FALLBACK_BDT_KEYS

print('BDT features:')
for k in BDT_FEATURE_KEYS:
    print('  ', k)


def load_cluster_feature_table(signal_h5, bib_h5, keys):
    tables = []
    for path in [signal_h5, bib_h5]:
        with h5py.File(path, 'r') as f:
            grp = f['clusters']
            missing = [k for k in keys if k not in grp]
            if missing:
                raise KeyError(f"Missing keys in {path}: {missing}")
            arr = np.stack([grp[k][:] for k in keys], axis=1).astype('float32')
            tables.append(arr)
    return np.concatenate(tables, axis=0)

X_bdt = load_cluster_feature_table(SIGNAL_H5, BIB_H5, BDT_FEATURE_KEYS)
print('X_bdt:', X_bdt.shape)
assert len(X_bdt) == len(labels)

X_train, y_train = X_bdt[idx_train], labels[idx_train]
X_val,   y_val   = X_bdt[idx_val],   labels[idx_val]
X_test,  y_test  = X_bdt[idx_test],  labels[idx_test]

## Quick feature checks

In [ ]:
df_features = pd.DataFrame(X_bdt, columns=BDT_FEATURE_KEYS)
df_features['label'] = labels

display(df_features.groupby('label')[BDT_FEATURE_KEYS].describe().T.head(40))

# Optional: use the tutorial helper if desired.
try:
    plot_feature_distributions(df_features, BDT_FEATURE_KEYS[:8], label_col='label')
except Exception as e:
    print('plot_feature_distributions helper not used:', e)

## Train or load the BDT

In [ ]:
TRAIN_BDT = not BDT_MODEL_PATH.exists()

if TRAIN_BDT:
    print('Training BDT...')
    t0 = time.time()
    bdt = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_SEED,
    )
    bdt.fit(X_train, y_train)
    joblib.dump(bdt, BDT_MODEL_PATH)
    print(f'Saved {BDT_MODEL_PATH}')
    print(f'Training time: {(time.time() - t0):.1f} s')
else:
    print(f'Loading {BDT_MODEL_PATH}')
    bdt = joblib.load(BDT_MODEL_PATH)

## Evaluate the BDT

In [ ]:
bdt_scores_val = bdt.predict_proba(X_val)[:, 1]
bdt_scores_test = bdt.predict_proba(X_test)[:, 1]

print('Validation AUC:', roc_auc_score(y_val, bdt_scores_val))
print('Test AUC:      ', roc_auc_score(y_test, bdt_scores_test))

bdt_results = evaluate_classifier(y_test, bdt_scores_test, 'BDT')
plot_score_distributions(y_test, bdt_scores_test, title='BDT score distribution')
plot_confusion_matrix(y_test, bdt_scores_test, cut=0.5, title='BDT confusion matrix, cut = 0.5')
plot_roc_comparison({'BDT': bdt_results})

## Feature importance

In [ ]:
if hasattr(bdt, 'feature_importances_'):
    importances = pd.DataFrame({
        'feature': BDT_FEATURE_KEYS,
        'importance': bdt.feature_importances_,
    }).sort_values('importance', ascending=False)
    display(importances)
    importances.to_csv(OUTPUT_DIR / 'bdt_feature_importance.csv', index=False)

    plt.figure(figsize=(8, 5))
    plt.barh(importances['feature'][::-1], importances['importance'][::-1])
    plt.xlabel('Feature importance')
    plt.title('BDT feature importance')
    plt.tight_layout()
    plt.show()
else:
    print('This BDT object does not expose feature_importances_.')

## Save a small BDT summary for the Transformer leaderboard

In [ ]:
def bib_rejection_at_signal_efficiency(y, scores, sig_eff):
    y = np.asarray(y).astype(int)
    scores = np.asarray(scores)
    sig_scores = scores[y == 1]
    bg_scores = scores[y == 0]
    threshold = np.quantile(sig_scores, 1.0 - sig_eff)
    bib_eff = np.mean(bg_scores >= threshold)
    return 1.0 - bib_eff

row = {
    'name': 'BDT',
    'kind': 'BDT',
    'test_auc': roc_auc_score(y_test, bdt_scores_test),
    'test_bib_rej_at_sig_eff_0p80': bib_rejection_at_signal_efficiency(y_test, bdt_scores_test, 0.80),
    'test_bib_rej_at_sig_eff_0p90': bib_rejection_at_signal_efficiency(y_test, bdt_scores_test, 0.90),
    'test_bib_rej_at_sig_eff_0p95': bib_rejection_at_signal_efficiency(y_test, bdt_scores_test, 0.95),
    'model_path': str(BDT_MODEL_PATH),
}
summary = pd.DataFrame([row])
display(summary)
summary.to_csv(OUTPUT_DIR / 'bdt_leaderboard_row.csv', index=False)